In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# QESEM: A Qiskit Function by Qedma
*See the [API reference](https://docs.quantum.ibm.com/api/functions/qedma-qesem)*

> **Note:** Qiskit Functions เป็นฟีเจอร์ทดลองที่ใช้ได้เฉพาะผู้ใช้ IBM Quantum&reg; Premium Plan, Flex Plan และ On-Prem (ผ่าน IBM Quantum Platform API) Plan เท่านั้น อยู่ในสถานะพรีวิวและอาจมีการเปลี่ยนแปลงได้

## ภาพรวม
แม้ว่า quantum processing units จะพัฒนาขึ้นอย่างมากในช่วงไม่กี่ปีที่ผ่านมา แต่ error จาก noise และความไม่สมบูรณ์ของ hardware ที่มีอยู่ยังคงเป็นความท้าทายหลักสำหรับนักพัฒนา quantum algorithm เมื่อสาขานี้เข้าใกล้การคำนวณควอนตัมระดับ utility-scale ที่ไม่สามารถตรวจสอบได้ด้วยคอมพิวเตอร์คลาสสิก การแก้ปัญหาสำหรับการยกเลิก noise ที่มีการรับประกันความแม่นยำกลายเป็นสิ่งสำคัญมากขึ้นเรื่อยๆ เพื่อเอาชนะความท้าทายนี้ Qedma ได้พัฒนา Quantum Error Mitigation (QESEM) ที่ผสานรวมอย่างราบรื่นบน IBM Quantum Platform เป็น [Qiskit Function](/guides/functions)

ด้วย QESEM ผู้ใช้สามารถรัน quantum circuits บน QPU ที่มี noise เพื่อรับผลลัพธ์ที่แม่นยำสูงและปราศจาก error พร้อม overhead ของเวลา QPU ที่มีประสิทธิภาพสูง ใกล้เคียงกับขอบเขตพื้นฐาน เพื่อให้บรรลุเป้าหมายนี้ QESEM ใช้ชุดวิธีการ proprietary ที่ Qedma พัฒนาขึ้นสำหรับการระบุลักษณะและลด error เทคนิคการลด error ได้แก่ gate optimization, noise-aware transpilation, error suppression (ES) และ unbiased error mitigation (EM) ด้วยการผสมผสานวิธีการที่อิงการระบุลักษณะเหล่านี้ ผู้ใช้สามารถได้ผลลัพธ์ที่เชื่อถือได้และปราศจาก error สำหรับ quantum circuits ปริมาณมากทั่วไป เปิดใช้งานแอปพลิเคชันที่ไม่สามารถทำได้อย่างอื่น

สำหรับคำอธิบายเต็มของ components พื้นฐานรวมถึงการสาธิตระดับ utility-scale ดูที่บทความ [Reliable high-accuracy error mitigation for utility-scale quantum circuits](https://arxiv.org/abs/2508.10997).
## คำอธิบาย
คุณสามารถใช้ฟังก์ชัน QESEM โดย Qedma เพื่อประเมินและรัน circuits ของคุณพร้อม error suppression และ mitigation ได้อย่างง่ายดาย ได้ circuit volumes ที่ใหญ่ขึ้นและความแม่นยำสูงขึ้น ในการใช้ QESEM คุณต้องให้ quantum circuit ชุด observables ที่จะวัด ความแม่นยำทางสถิติเป้าหมายสำหรับแต่ละ observable และ QPU ที่เลือก ก่อนรัน circuit ตามความแม่นยำเป้าหมาย คุณสามารถประเมินเวลา QPU ที่ต้องการโดยอิงการคำนวณเชิงวิเคราะห์ที่ไม่ต้องรัน circuit เมื่อพอใจกับการประเมินเวลา QPU แล้ว คุณก็สามารถรัน circuit ด้วย QESEM ได้

เมื่อคุณรัน circuit, QESEM จะรัน device characterization protocol ที่ปรับแต่งเฉพาะสำหรับ circuit ของคุณ ให้โมเดล noise ที่เชื่อถือได้สำหรับ error ที่เกิดขึ้นใน circuit จากการระบุลักษณะ QESEM จะ implement noise-aware transpilation ก่อน เพื่อ map input circuit ลงบน physical qubits และ gates ที่ลด noise ที่ส่งผลต่อ target observable ซึ่งรวมถึง gates ที่มีอยู่ native (CX/CZ บน IBM&reg; devices) รวมถึง gates เพิ่มเติมที่ QESEM optimize ก่อตัวเป็น extended gate set ของ QESEM จากนั้น QESEM รันชุด ES และ EM circuits ที่อิงการระบุลักษณะบน QPU และเก็บผล measurement รูทีนเหล่านี้ถูก post-process แบบ classically เพื่อให้ค่า expectation value ที่ไม่มี bias และ error bar สำหรับแต่ละ observable ตามความแม่นยำที่ร้องขอ

![ภาพรวม Qedma QESEM](../docs/images/guides/qedma-qesem/overview.svg)
QESEM ได้รับการสาธิตว่าให้ผลลัพธ์ที่มีความแม่นยำสูงสำหรับ quantum applications หลากหลายรูปแบบและบน circuit volumes ที่ใหญ่ที่สุดที่ทำได้ในปัจจุบัน QESEM มีฟีเจอร์ต่อไปนี้ที่แสดงในส่วน benchmarks ด้านล่าง:
-	**ความแม่นยำที่รับประกัน:** QESEM ให้ผลลัพธ์ประมาณค่า expectation values ของ observables ที่ไม่มี bias วิธี EM มีการรับประกันทางทฤษฎี ซึ่งร่วมกับการระบุลักษณะระดับ cutting-edge ของ Qedma รับประกันว่า mitigation จะ converge ไปสู่ noiseless circuit output ในระดับความแม่นยำที่ผู้ใช้กำหนด ต่างจากวิธี EM แบบ heuristic หลายวิธีที่เสี่ยงต่อ systematic errors หรือ biases ความแม่นยำที่รับประกันของ QESEM เป็นสิ่งสำคัญสำหรับให้ผลลัพธ์ที่เชื่อถือได้ใน quantum circuits และ observables ทั่วไป
-	**ขยายได้สำหรับ QPU ขนาดใหญ่:** เวลา QPU ของ QESEM ขึ้นอยู่กับ circuit volumes แต่เป็นอิสระจากจำนวน Qubit Qedma ได้สาธิต QESEM บน quantum devices ที่ใหญ่ที่สุดที่มีอยู่ในปัจจุบัน รวมถึง IBM Quantum Eagle 127-qubit และ Heron 133-qubit devices
-	**ไม่เฉพาะเจาะจงกับแอปพลิเคชัน:** QESEM ได้รับการสาธิตในแอปพลิเคชันหลากหลาย รวมถึง Hamiltonian simulation, VQE, QAOA และ amplitude estimation ผู้ใช้สามารถป้อน quantum circuit และ observable ใดๆ ที่ต้องการวัดและได้ผลลัพธ์ที่แม่นยำและปราศจาก error ข้อจำกัดเดียวถูกกำหนดโดย hardware specifications และเวลา QPU ที่จัดสรรไว้ ซึ่งกำหนด circuit volumes และความแม่นยำของ output ที่เข้าถึงได้ ต่างจากโซลูชันลด error หลายตัวที่เฉพาะเจาะจงกับแอปพลิเคชันหรือใช้ heuristics ที่ไม่ควบคุม ทำให้ใช้ไม่ได้กับ quantum circuits และแอปพลิเคชันทั่วไป
-  **Extended gate set:** QESEM รองรับ fractional-angle gates และมี Qedma-optimized fractional-angle $Rzz(\theta)$ gates บน IBM Quantum Heron และ Eagle devices Extended gate set นี้ช่วยให้ compilation มีประสิทธิภาพมากขึ้นและเปิดล็อก circuit volumes ที่ใหญ่กว่าถึง 2 เท่าเมื่อเทียบกับ default CX/CZ compilation
-	**Multibase observables:** QESEM รองรับ input observables ที่ประกอบด้วย Pauli strings ที่ไม่ commute กันหลายตัว เช่น generic Hamiltonians การเลือก measurement bases และการ optimize การจัดสรรทรัพยากร QPU (shots และ circuits) จะทำโดยอัตโนมัติโดย QESEM เพื่อลดเวลา QPU ที่ต้องการสำหรับความแม่นยำที่ร้องขอ การ optimize นี้ ซึ่งคำนึงถึง hardware fidelities และ execution rates ช่วยให้คุณรัน circuits ที่ลึกขึ้นและได้ความแม่นยำสูงขึ้น
## Benchmarks
QESEM ได้รับการทดสอบกับ use cases และแอปพลิเคชันหลากหลาย ตัวอย่างต่อไปนี้ช่วยให้คุณประเมินได้ว่า workloads ประเภทใดที่คุณสามารถรันกับ QESEM ได้

ตัวชี้วัดสำคัญสำหรับการวัดความยากของทั้ง error mitigation และ classical simulation สำหรับ circuit และ observable ที่กำหนดคือ **active volume**: จำนวน CNOT gates ที่ส่งผลต่อ observable ใน circuit active volume ขึ้นอยู่กับความลึกและความกว้างของ circuit, น้ำหนักของ observable และโครงสร้างของ circuit ซึ่งกำหนด light cone ของ observable สำหรับรายละเอียดเพิ่มเติม ดูการบรรยายจาก [2024 IBM Quantum Summit](https://www.youtube.com/watch?v=Hd-IGvuARfE&t=1730s) QESEM มีประโยชน์อย่างมากในระดับ high-volume โดยให้ผลลัพธ์ที่เชื่อถือได้สำหรับ circuits และ observables ทั่วไป

![Active volume](../docs/images/guides/qedma-qesem/active_volume.svg)

| แอปพลิเคชัน    | จำนวน Qubit | Device | คำอธิบาย Circuit | ความแม่นยำ | เวลาทั้งหมด | การใช้งาน Runtime |
| ---------  | ---------------- | ----- | -------------------------- | -------- | ---------- | ------------- |
| VQE circuit  | 8              | Eagle (r3) | 21 layers ทั้งหมด, 9 measurement bases, 1D chain                    | 98%      | 35 นาที      | 14 นาที         |
| Kicked Ising   | 28              | Eagle (r3) | 3 unique layers x 3 steps, 2D heavy-hex topology                      | 97%     | 22 นาที      | 4 นาที          |
| Kicked Ising   | 28              | Eagle (r3) | 3 unique layers x 8 steps, 2D heavy-hex topology                     | 97%      | 116 นาที      | 23 นาที          |
| Trotterized Hamiltonian simulation   | 40  | Eagle (r3)            | 2 unique layers x 10 Trotter steps, 1D chain                    | 97%      | 3 ชั่วโมง     | 25 นาที         |
| Trotterized Hamiltonian simulation   | 119  | Eagle (r3)           | 3 unique layers x 9 Trotter steps, 2D heavy-hex topology                    | 95%      | 6.5 ชั่วโมง     | 45 นาที         |
| Kicked Ising  | 136             | Heron (r2) | 3 unique layers x 15 steps, 2D heavy-hex topology                    | 99%      | 52 นาที             | 9 นาที           |

ความแม่นยำวัดที่นี่เทียบกับค่า ideal ของ observable: $\frac{\langle O \rangle_{ideal} - \epsilon}{\langle O \rangle_{ideal}}$ โดยที่ '$\epsilon$' คือความแม่นยำสัมบูรณ์ของ mitigation (กำหนดโดย user input) และ $\langle O \rangle_{ideal}$ คือ observable บน noiseless circuit
'Runtime usage' วัดการใช้งาน benchmark ในโหมด batch (ผลรวมการใช้งานของแต่ละ job) ในขณะที่ 'total time' วัดการใช้งานในโหมด session (experiment wall time) ซึ่งรวมเวลา classical และ communication เพิ่มเติม QESEM พร้อมใช้งานในทั้งสองโหมด เพื่อให้ผู้ใช้ใช้ทรัพยากรที่มีอยู่ได้อย่างดีที่สุด

28-qubit Kicked Ising circuits จำลอง Discrete Time Quasicrystal ที่ศึกษาโดย Shinjo et al. (ดู [arXiv 2403.16718](https://arxiv.org/abs/2403.16718) และ [Q2B24 Tokyo](https://www.youtube.com/watch?v=tQW6FdLc6zo)) บน three connected loops ของ ibm_kawasaki พารามิเตอร์ circuit ที่ใช้คือ $(\theta_x, \theta_z) = (0.9 \pi, 0)$ พร้อม ferromagnetic initial state $| \psi_0 \rangle = | 0 \rangle ^{\otimes n}$ observable ที่วัดคือค่าสัมบูรณ์ของ magnetization $M = |\frac{1}{28} \sum_{i=0}^{27} \langle Z_i \rangle|$ การทดลอง utility-scale Kicked Ising รันบน 136 qubits ที่ดีที่สุดของ ibm_fez benchmark เฉพาะนี้รันที่ Clifford angle $(\theta_x, \theta_z) = (\pi, 0)$ ซึ่ง active volume โตช้าตาม circuit depth ซึ่งร่วมกับ device fidelities สูงทำให้ได้ความแม่นยำสูงใน runtime สั้น

Trotterized Hamiltonian simulation circuits สำหรับ Transverse-Field Ising model ที่ fractional angles: $(\theta_{zz}, \theta_x) = (\pi / 4, \pi /8)$ และ $(\theta_{zz}, \theta_x) = (\pi / 6, \pi / 8)$ ตามลำดับ (ดู [Q2B24 Tokyo](https://www.youtube.com/watch?v=tQW6FdLc6zo)) utility-scale circuit รันบน 119 qubits ที่ดีที่สุดของ ibm_brisbane ในขณะที่การทดลอง 40-qubit รันบน chain ที่ดีที่สุดที่มีอยู่ ความแม่นยำรายงานสำหรับ magnetization ผลลัพธ์ความแม่นยำสูงได้รับสำหรับ observables ที่มีน้ำหนักสูงกว่าด้วย

VQE circuit ถูกพัฒนาร่วมกับนักวิจัยจาก Center for Quantum Technology and Applications ที่ Deutsches Elektronen-Synchrotron (DESY) target observable ที่นี่คือ Hamiltonian ที่ประกอบด้วย Pauli strings ที่ไม่ commute กันจำนวนมาก เน้นประสิทธิภาพที่ optimize ของ QESEM สำหรับ multi-basis observables Mitigation ถูกนำไปใช้กับ classically-optimized ansatz แม้ว่าผลลัพธ์เหล่านี้ยังไม่ได้ตีพิมพ์ ผลลัพธ์คุณภาพเดียวกันจะได้รับสำหรับ circuits ต่างๆ ที่มีคุณสมบัติโครงสร้างคล้ายกัน
## เริ่มต้นใช้งาน
ยืนยันตัวตนโดยใช้ [IBM Quantum Platform API key](http://quantum.cloud.ibm.com/) และเลือก QESEM Qiskit Function ดังนี้ (snippet นี้สมมติว่าคุณได้ [บันทึกบัญชีของคุณ](/guides/functions#install-qiskit-functions-catalog-client) ไว้ใน local environment แล้ว)

In [1]:
import qiskit

from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

qesem_function = catalog.load("qedma/qesem")

## ตัวอย่าง
เพื่อเริ่มต้น ลองตัวอย่างพื้นฐานนี้ในการประเมินเวลา QPU ที่ต้องการในการรัน QESEM สำหรับ `pub` ที่กำหนด:

In [ ]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend_name = service.least_busy().name

In [ ]:
circ = qiskit.QuantumCircuit(5)
circ.cx(0, 1)
circ.cx(2, 3)
circ.cx(1, 2)
circ.cx(3, 4)

avg_magnetization = qiskit.quantum_info.SparsePauliOp.from_sparse_list(
    [("Z", [q], 1 / 5) for q in range(5)], num_qubits=5
)
other_observable = qiskit.quantum_info.SparsePauliOp.from_sparse_list(
    [("ZZ", [0, 1], 1.0), ("XZ", [1, 4], 0.5)], num_qubits=5
)

time_estimation_job = qesem_function.run(
    pubs=[(circ, [avg_magnetization, other_observable])],
    options={
        "estimate_time_only": "analytical",
    },
    backend_name=backend_name,  # E.g. "ibm_fez"
)

time_estimate_result = time_estimation_job.result()

ตัวอย่างต่อไปนี้รัน QESEM job:

In [ ]:
sample_job = qesem_function.run(
    pubs=[(circ, [avg_magnetization, other_observable])],
    backend_name=backend_name,  # E.g. "ibm_fez"
)

คุณสามารถใช้ Qiskit Serverless APIs ที่คุ้นเคยเพื่อตรวจสอบสถานะหรือดึงผลลัพธ์ของ Qiskit Function workload:

In [ ]:
print(sample_job.status())
result = sample_job.result()

code snippet ต่อไปนี้อธิบายวิธีดึงการประเมินเวลา QPU (เมื่อตั้ง `estimate_time_only`):

In [ ]:
print(
    f"The estimated QPU time for this PUB is: "
    f"\n{time_estimate_result[0].metadata}"
)

code snippet ต่อไปนี้แสดงวิธีดึงผลลัพธ์ mitigation (เมื่อไม่ได้ตั้ง `estimate_time_only`) และ execution metrics ซึ่งประกอบด้วยข้อมูลสำคัญที่ช่วยให้เข้าใจได้ลึกขึ้นว่าพารามิเตอร์ต่างๆ ส่งผลต่อการรัน QESEM อย่างไร อาจมีความเกี่ยวข้องเมื่อเขียนบทความโดยอิงงานวิจัยของคุณด้วย

In [ ]:
results = result[0]
print(f"Mitigated expectation values: \n{results.data.evs}")
print(f"Mitigated error-bar: \n{results.data.stds}")
noisy_results = results.metadata["noisy_results"]
print(f"Noisy expectation values: \n{noisy_results.evs}")
print(f"Noisy error-bar: \n{noisy_results.stds}")
print(f"Total QPU time: \n {results.metadata['total_qpu_time']}")
print(
    f"Gates fidelity measured during the experiment: "
    f"\n {results.metadata['gate_fidelities']}"
)
print(
    f"Total shots / mitigation shots: \n "
    f"{results.metadata['total_shots']} / "
    f"{results.metadata['mitigation_shots']}"
)
print("Transpiled circuits:")
for i, circuit in enumerate(results.metadata["transpiled_circs"]):
    print(f"Circuit {i}:")
    print(f"  Circuit: \n {circuit['circuit']}")
    print(f"  Qubit mapping: \n {circuit['qubit_map']}")
    print(f"  Measurement bases: \n {circuit['num_measurement_bases']}")

## ดึงข้อความ error
หาก workload status เป็น ERROR ให้ใช้ `job.result()` เพื่อดึงข้อความ error ดังนี้:

In [ ]:
print(sample_job.result())

PrimitiveResult([PubResult(data=DataBin(), metadata={'time_estimation_sec': 12600})], metadata={})


## Get support

The Qedma support team is here to help! If you encounter any issues or have questions about using the QESEM Qiskit Function, please don't hesitate to reach out. Our knowledgeable and friendly support staff are ready to assist you with any technical concerns or inquiries you may have.

You can email us at support@qedma.com for assistance. Please include as much detail as possible about the issue you're experiencing to help us provide a swift and accurate response. You can also contact your dedicated Qedma POC representative via email or phone.

To help us assist you more efficiently, please provide the following information when you contact us:

- A detailed description of the issue
- The job ID
- Any relevant error messages or codes


We are committed to providing you with prompt and effective support to ensure you have the best possible experience with our Qiskit Function.

We are always looking to improve our product and we welcome your suggestions! If you have ideas on how we can enhance our services or features you'd like to see, please send us your thoughts at support@qedma.com.

## Next steps

<Admonition type="tip" title="Recommendations">

- [Request access to Qedma QESEM](https://quantum.cloud.ibm.com/functions?id=qedma-qesem).
- Visit the [API reference](/docs/api/functions/qedma-qesem) for this Qiskit Function.
- Review [Bauman, N. P., et al. (2025). Coupled Cluster Downfolding Theory in Simulations of Chemical Systems on Quantum Hardware. arXiv preprint arXiv:2507.01199](https://arxiv.org/pdf/2507.01199).
- Review [Aharonov, D., et al. (2025). Reliable high-accuracy error mitigation for utility-scale quantum circuits. arXiv preprint arXiv:2508.10997](https://arxiv.org/pdf/2508.10997).


</Admonition>